<a href="https://colab.research.google.com/github/Jenny5789/crop-image-classification/blob/main/notebooks/development/crop_classifier_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ===================================
# 2부: 데이터 재분배 이후 재검증
# ===================================

앞선 실험(1~7)은 AI Hub 원본의 Training/Validation 구분을 유지한 상태에서
샘플링한 데이터를 사용함.

이후 실제 이미지를 비교하는 과정에서 두 세트 간 피사체 및 생육 단계의
구성 차이가 관찰되어, 기존 Training과 Validation 데이터를 통합한 뒤
무작위로 재분할함.

재분배 이후에는 동일한 모델 조건으로 실험을 진행하여
데이터 분할 변경에 따른 성능 변화를 확인함.

In [ ]:
import os
import random
import shutil
import time
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from tensorflow import keras
from tensorflow.keras import models, layers

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

print(IMG_SIZE, BATCH_SIZE)

128 32


In [ ]:
import os
import random
import shutil

raw_path = r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\raw"
reprocessed_path = r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\reprocessed"

crop_map = {
    "고추": "pepper",
    "배추": "cabbage",
    "오이": "cucumber"
}

random.seed(42)

for crop_kor, crop_eng in crop_map.items():
    train_folder = os.path.join(raw_path, "Training", f"[원천]{crop_kor}_0.정상")
    val_folder = os.path.join(raw_path, "Validation", f"[원천]{crop_kor}_0.정상")

    train_files = [(f, train_folder) for f in os.listdir(train_folder)]
    val_files = [(f, val_folder) for f in os.listdir(val_folder)]

    all_files = train_files + val_files
    random.shuffle(all_files)

    print(f"{crop_kor}: 전체 {len(all_files)}장 (Train {len(train_files)} + Val {len(val_files)})")

    n_train = 600
    n_val = 150

    selected = random.sample(all_files, n_train + n_val)
    new_train = selected[:n_train]
    new_val = selected[n_train:]

    dst_train = os.path.join(reprocessed_path, "train", crop_eng)
    dst_val = os.path.join(reprocessed_path, "val", crop_eng)
    os.makedirs(dst_train, exist_ok=True)
    os.makedirs(dst_val, exist_ok=True)

    for fname, src_folder in new_train:
        shutil.copy2(
            os.path.join(src_folder, fname),
            os.path.join(dst_train, fname)
        )

    for fname, src_folder in new_val:
        shutil.copy2(
            os.path.join(src_folder, fname),
            os.path.join(dst_val, fname)
        )

    print(f"  -> train {len(new_train)}장, val {len(new_val)}장 복사 완료")

print("\n전체 재분배 완료!")

고추: 전체 2705장 (Train 1595 + Val 1110)
  -> train 600장, val 150장 복사 완료
배추: 전체 2008장 (Train 825 + Val 1183)
  -> train 600장, val 150장 복사 완료
오이: 전체 2227장 (Train 623 + Val 1604)
  -> train 600장, val 150장 복사 완료

전체 재분배 완료!


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\reprocessed\train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\reprocessed\val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int'
)

class_names = train_ds.class_names

print("클래스 목록:", class_names)

Found 1800 files belonging to 3 classes.
Found 450 files belonging to 3 classes.
클래스 목록: ['cabbage', 'cucumber', 'pepper']


In [ ]:
import time

experiment_results = []

def run_experiment(name, model, train_ds, val_ds, epochs=15, verbose=1):
    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=verbose
    )

    elapsed = time.time() - start_time

    experiment_results.append({
        "실험명": name,
        "epochs": epochs,
        "총_파라미터": model.count_params(),
        "train_acc": round(history.history['accuracy'][-1], 4),
        "val_acc": round(history.history['val_accuracy'][-1], 4),
        "train_loss": round(history.history['loss'][-1], 4),
        "val_loss": round(history.history['val_loss'][-1], 4),
        "학습시간_초": round(elapsed, 1)
    })

    print(f"[기록 완료] {name}")

    return history

In [ ]:
model_v2_check = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model_v2_check.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_v2_check = run_experiment(
    "simplified_model on rebalanced_data",
    model_v2_check,
    train_ds,
    val_ds,
    epochs=15
)

Epoch 1/15


C:\Users\AISW_203_103\anaconda3\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


57/57 ━━━━━━━━━━━━━━━━━━━━ 37s 610ms/step - accuracy: 0.3989 - loss: 1.0720 - val_accuracy: 0.6333 - val_loss: 0.9658
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 427ms/step - accuracy: 0.5322 - loss: 0.9222 - val_accuracy: 0.6600 - val_loss: 0.8265
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 418ms/step - accuracy: 0.6350 - loss: 0.8253 - val_accuracy: 0.6422 - val_loss: 0.7507
Epoch 4/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 425ms/step - accuracy: 0.6694 - loss: 0.7590 - val_accuracy: 0.6867 - val_loss: 0.6825
Epoch 5/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 418ms/step - accuracy: 0.6967 - loss: 0.7210 - val_accuracy: 0.7400 - val_loss: 0.6413
Epoch 6/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 47s 513ms/step - accuracy: 0.7117 - loss: 0.6851 - val_accuracy: 0.7200 - val_loss: 0.6221
Epoch 7/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 439ms/step - accuracy: 0.7222 - loss: 0.6631 - val_accuracy: 0.7200 - val_loss: 0.6925
Epoch 8/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 409ms/step - accuracy: 0.7400 - loss: 0.6416 - val_accuracy: 0.737

## 실험 8: 데이터 재분배 후 재검증 결과 분석 (실험 5와 동일 모델 구조)

- train accuracy: 0.3989 → 0.8111
- val_accuracy: 0.6333 → 0.7978 (학습 중 최고 0.8089)
- val_loss: 0.9658 → 0.4490
- **실험 5(재분배 전 데이터)와 비교**:
  - train_acc: 0.9200 → 0.8111 (하락)
  - val_acc: 0.6511 → 0.7978 (상승)
  - val_loss: 1.0065 → 0.4490 (감소)
  - train-val accuracy 격차: 26.89%p → 1.33%p로 축소
- **결론**: 데이터 재분배 후 동일한 모델 구조로 다시 학습한 결과, 실험 5보다 val_accuracy가 상승하고 val_loss가 감소함. Train/Validation의 데이터 구성을 변경한 조건에서 Validation 성능이 개선됨.

## 실험 1~8 흐름 요약

| 실험 | 적용한 내용 | 결과 |
|---|---|---|
| 실험 1 | Baseline CNN | Train 성능만 크게 상승 → **과적합 확인** |
| 실험 2 | Data Augmentation 적용 | Validation 성능 **개선되지 않음** |
| 실험 3 | Dropout 0.3 → 0.5 | Baseline보다 Validation 성능 **일부 개선** |
| 실험 4 | Data Augmentation + Dropout 0.5 | Validation 성능 **악화** |
| 실험 5 | 필터 축소 + GAP 적용으로 모델 단순화 | 과적합 격차와 Val Loss **감소** → 새로운 기준 모델 |
| 실험 6 | 단순화 모델 + Data Augmentation | 실험 5보다 Validation 성능 **하락** |
| 실험 7 | 모델 추가 단순화 | 파라미터 감소 + Validation 성능 **소폭 개선** |
| 실험 8 | Train/Validation 데이터 재분배 후 실험 5 모델 재검증 | Validation 성능 **크게 개선**, Train-Val 격차 **크게 감소** |

In [ ]:
model_v2_longer = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])
model_v2_longer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_v2_longer = run_experiment("simplified_model on rebalanced_data (epoch30)", model_v2_longer, train_ds, val_ds, epochs=30)

Epoch 1/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 27s 439ms/step - accuracy: 0.4917 - loss: 1.0085 - val_accuracy: 0.5778 - val_loss: 0.9019
Epoch 2/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 440ms/step - accuracy: 0.5578 - loss: 0.9027 - val_accuracy: 0.6422 - val_loss: 0.8231
Epoch 3/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 41s 434ms/step - accuracy: 0.6072 - loss: 0.8178 - val_accuracy: 0.6000 - val_loss: 0.8876
Epoch 4/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 443ms/step - accuracy: 0.6511 - loss: 0.7767 - val_accuracy: 0.5867 - val_loss: 0.8085
Epoch 5/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 29s 507ms/step - accuracy: 0.6917 - loss: 0.7139 - val_accuracy: 0.6267 - val_loss: 0.7454
Epoch 6/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 27s 462ms/step - accuracy: 0.6967 - loss: 0.6859 - val_accuracy: 0.7356 - val_loss: 0.5986
Epoch 7/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 427ms/step - accuracy: 0.7333 - loss: 0.6238 - val_accuracy: 0.7533 - val_loss: 0.5785
Epoch 8/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 434ms/step - accuracy: 0.7311 - loss: 0.6447 - val_accu

## 실험 9: 재분배 데이터 + Epoch 30 결과 분석

- train accuracy: 0.4917 → 0.8439
- val_accuracy: 0.5778 → 0.8267 (학습 중 최고 0.8444, Epoch 23)
- val_loss: 0.9019 → 0.4208 (학습 중 최저 0.3569, Epoch 28)

- **실험 8(재분배 데이터, Epoch 15)과 비교**:
  - train_acc: 0.8111 → 0.8439 (상승)
  - val_acc: 0.7978 → 0.8267 (상승)
  - val_loss: 0.4490 → 0.4208 (감소)

- **결론**: 동일한 모델 구조와 재분배 데이터를 사용하고 Epoch 설정을 15에서 30으로 늘린 결과, 최종 train accuracy와 val_accuracy가 상승하고 val_loss가 감소함. 이번 실행에서는 Epoch을 늘린 조건에서 Validation 성능이 추가로 개선됨.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import time

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

model_v2_epoch50 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model_v2_epoch50.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start_time = time.time()

history_v2_epoch50 = model_v2_epoch50.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

elapsed = time.time() - start_time

# 실제 학습된 Epoch 수
actual_epochs = len(history_v2_epoch50.history['accuracy'])

# EarlyStopping이 복원한 최종 모델을 다시 평가
train_loss, train_acc = model_v2_epoch50.evaluate(
    train_ds,
    verbose=0
)

val_loss, val_acc = model_v2_epoch50.evaluate(
    val_ds,
    verbose=0
)

experiment_results.append({
    "실험명": "simplified_model on rebalanced_data (epoch50, early_stop)",
    "epochs": actual_epochs,
    "총_파라미터": model_v2_epoch50.count_params(),
    "train_acc": round(train_acc, 4),
    "val_acc": round(val_acc, 4),
    "train_loss": round(train_loss, 4),
    "val_loss": round(val_loss, 4),
    "학습시간_초": round(elapsed, 1)
})

print("[기록 완료] simplified_model on rebalanced_data (epoch50, early_stop)")
print("실제 학습 Epoch:", actual_epochs)
print("복원 모델 Train Accuracy:", round(train_acc, 4))
print("복원 모델 Validation Accuracy:", round(val_acc, 4))
print("복원 모델 Validation Loss:", round(val_loss, 4))

Epoch 1/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 26s 425ms/step - accuracy: 0.4617 - loss: 1.0322 - val_accuracy: 0.5711 - val_loss: 0.8902
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 430ms/step - accuracy: 0.5678 - loss: 0.8773 - val_accuracy: 0.6622 - val_loss: 0.7816
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 436ms/step - accuracy: 0.6239 - loss: 0.8199 - val_accuracy: 0.6778 - val_loss: 0.7575
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 428ms/step - accuracy: 0.6222 - loss: 0.8327 - val_accuracy: 0.7022 - val_loss: 0.7230
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 27s 470ms/step - accuracy: 0.6633 - loss: 0.7620 - val_accuracy: 0.6178 - val_loss: 0.7545
Epoch 6/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 444ms/step - accuracy: 0.6628 - loss: 0.7472 - val_accuracy: 0.7067 - val_loss: 0.6729
Epoch 7/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 436ms/step - accuracy: 0.6989 - loss: 0.6828 - val_accuracy: 0.6933 - val_loss: 0.6992
Epoch 8/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 423ms/step - accuracy: 0.7089 - loss: 0.6950 - val_accu

In [ ]:
import os

model_dir = r"C:\Users\AISW_203_103\Desktop\crop-classifier\model"
os.makedirs(model_dir, exist_ok=True)

model_v2_epoch50.save(
    os.path.join(model_dir, "crop_classifier_final_600.keras")
)

print("저장 완료: crop_classifier_final_600.keras")

저장 완료: crop_classifier_final_600.keras


## 실험 10: 재분배 데이터 + Epoch 50(EarlyStopping) 결과 분석

- EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True) 적용
- 최대 50 Epoch까지 설정했으며, val_loss가 계속 개선되어 50 Epoch 모두 학습함
- 학습 중 최고 val_accuracy: 0.8733 (Epoch 49)
- 학습 중 최저 val_loss: 0.3261 (Epoch 50)
- restore_best_weights=True에 따라 최저 val_loss 시점인 Epoch 50의 가중치가 최종 모델에 적용됨
- 복원 모델 evaluate 결과:
  - train_accuracy: 0.8722
  - val_accuracy: 0.8644
  - val_loss: 0.3261
- 복원 모델을 `crop_classifier_final_600.keras`로 저장함
- **결론**: Epoch을 최대 50까지 확대하고 EarlyStopping을 적용한 결과, 조기 종료 없이 Epoch 50까지 학습이 진행됨. 실험 9보다 Validation Accuracy가 상승하고 Validation Loss가 감소했으며, Epoch 50에서 가장 낮은 val_loss를 기록함.

In [ ]:
df_results = pd.DataFrame(experiment_results)
df_results.to_csv(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\experiment_results.csv",
    index=False,
    encoding="utf-8-sig"
)

df_results

,실험명,epochs,총_파라미터,train_acc,val_acc,train_loss,val_loss,학습시간_초
0,simplified_model on rebalanced_data,15,15491,0.8111,0.7978,0.4784,0.4490,404.5
1,simplified_model on rebalanced_data (epoch30),30,15491,0.8439,0.8267,0.3998,0.4208,811.2
2,"simplified_model on rebalanced_data (epoch50, ...",50,15491,0.8722,0.8644,0.3048,0.3261,1270.7


In [ ]:
csv_path = r"C:\Users\AISW_203_103\Desktop\crop-classifier\experiment_results.csv"

df_results = pd.read_csv(csv_path)
experiment_results = df_results.to_dict('records')

df_results

,실험명,epochs,총_파라미터,train_acc,val_acc,train_loss,val_loss,학습시간_초
0,simplified_model on rebalanced_data,15,15491,0.8111,0.7978,0.4784,0.4490,404.5
1,simplified_model on rebalanced_data (epoch30),30,15491,0.8439,0.8267,0.3998,0.4208,811.2
2,"simplified_model on rebalanced_data (epoch50, ...",50,15491,0.8722,0.8644,0.3048,0.3261,1270.7


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import time

early_stop_smaller = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

model_v2_smaller = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model_v2_smaller.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start_time = time.time()

history_v2_smaller = model_v2_smaller.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop_smaller],
    verbose=1
)

elapsed = time.time() - start_time

actual_epochs = len(history_v2_smaller.history['accuracy'])

train_loss, train_acc = model_v2_smaller.evaluate(
    train_ds,
    verbose=0
)

val_loss, val_acc = model_v2_smaller.evaluate(
    val_ds,
    verbose=0
)

experiment_results.append({
    "실험명": "smaller_model on rebalanced_data (epoch50, early_stop)",
    "epochs": actual_epochs,
    "총_파라미터": model_v2_smaller.count_params(),
    "train_acc": round(train_acc, 4),
    "val_acc": round(val_acc, 4),
    "train_loss": round(train_loss, 4),
    "val_loss": round(val_loss, 4),
    "학습시간_초": round(elapsed, 1)
})

print("[기록 완료] smaller_model on rebalanced_data (epoch50, early_stop)")
print("실제 학습 Epoch:", actual_epochs)
print("복원 모델 Train Accuracy:", round(train_acc, 4))
print("복원 모델 Validation Accuracy:", round(val_acc, 4))
print("복원 모델 Validation Loss:", round(val_loss, 4))

Epoch 1/50


C:\Users\AISW_203_103\anaconda3\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


57/57 ━━━━━━━━━━━━━━━━━━━━ 26s 415ms/step - accuracy: 0.4094 - loss: 1.0679 - val_accuracy: 0.5133 - val_loss: 1.0113
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 26s 450ms/step - accuracy: 0.5189 - loss: 0.9571 - val_accuracy: 0.6400 - val_loss: 0.8592
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 29s 513ms/step - accuracy: 0.5456 - loss: 0.9159 - val_accuracy: 0.6622 - val_loss: 0.8363
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 27s 479ms/step - accuracy: 0.5978 - loss: 0.8535 - val_accuracy: 0.6956 - val_loss: 0.7681
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 30s 516ms/step - accuracy: 0.6294 - loss: 0.8052 - val_accuracy: 0.6244 - val_loss: 0.8458
Epoch 6/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 409ms/step - accuracy: 0.6483 - loss: 0.7934 - val_accuracy: 0.7267 - val_loss: 0.7058
Epoch 7/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 41s 410ms/step - accuracy: 0.6706 - loss: 0.7656 - val_accuracy: 0.7000 - val_loss: 0.7036
Epoch 8/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 406ms/step - accuracy: 0.6667 - loss: 0.7535 - val_accuracy: 0.500

## 실험 11: 재분배 데이터 + 추가 모델 단순화(파라미터 5,411개) 결과 분석

- train accuracy: 0.4094 → 복원 모델 0.8400
- val_accuracy: 0.5133 → 복원 모델 0.8267
- val_loss: 1.0113 → 복원 모델 0.4246
- 최대 50 epoch 중 40 epoch에서 EarlyStopping으로 학습 종료
- 학습 중 최고 val_accuracy: 0.8333 (epoch 32, 39, 40)
- 학습 중 최저 val_loss: 0.4246 (epoch 33)
- restore_best_weights=True에 따라 epoch 33의 가중치가 복원됨

- **실험 10(파라미터 15,491개)과 비교**:
  - 파라미터 수: 15,491 → 5,411 (약 65% 감소)
  - val_accuracy: 0.8644 → 0.8267
  - val_loss: 0.3261 → 0.4246
  - Train-Val Accuracy 격차: 약 0.78%p → 약 1.33%p
  - 실험 10은 epoch 50까지 val_loss가 개선된 반면, 실험 11은 epoch 33 이후 최저 val_loss가 갱신되지 않아 epoch 40에서 EarlyStopping됨

- **분석**:
  - 재분배 이후 실험 10에서는 이미 Train/Validation Accuracy 격차가 크지 않았음
  - 따라서 추가적인 모델 단순화가 과적합 완화에 반드시 필요한 상태는 아니었음
  - 파라미터를 약 65% 추가 감소시켰지만 Validation Accuracy가 감소하고 Validation Loss가 증가함
  - 추가 단순화 모델에서는 Validation Loss 개선도 더 일찍 멈춤

- **결론**: 모델 복잡도를 낮추는 것이 항상 Validation 성능 개선으로 이어지는 것은 아니었음. 이번 조건에서는 5,411개까지 추가 단순화하기보다 15,491개 파라미터의 실험 10 구조가 더 높은 Validation 성능을 보였음.

# ===================================
# 3부: DNN vs CNN 비교
# ===================================

이미지의 공간적 특징을 추출하는 CNN과 이미지를 1차원으로 펼쳐
완전연결층(Dense)으로 학습하는 DNN의 차이를 실제 결과에서 확인하기 위해,
동일한 재분배 데이터(train_ds, val_ds)를 사용하여 두 구조의 성능을 비교함.

CNN은 실험 10의 모델을 비교 기준으로 사용하고,
DNN은 동일한 입력 데이터와 학습 조건에서 새로 학습함.

In [ ]:
model_dnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Rescaling(1./255),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model_dnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_dnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)              │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 49152)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │       1,572,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,572,995 (6.00 MB)

 Trainable params: 1,572,995 (6.00 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import time

early_stop_dnn = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

start_time = time.time()

history_dnn = model_dnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop_dnn],
    verbose=1
)

elapsed_dnn = time.time() - start_time

actual_epochs_dnn = len(history_dnn.history['loss'])

train_loss_dnn, train_acc_dnn = model_dnn.evaluate(
    train_ds,
    verbose=0
)

val_loss_dnn, val_acc_dnn = model_dnn.evaluate(
    val_ds,
    verbose=0
)

best_val_acc_dnn = max(history_dnn.history['val_accuracy'])
best_val_acc_epoch_dnn = (
    history_dnn.history['val_accuracy'].index(best_val_acc_dnn) + 1
)

best_val_loss_dnn = min(history_dnn.history['val_loss'])
best_val_loss_epoch_dnn = (
    history_dnn.history['val_loss'].index(best_val_loss_dnn) + 1
)

print("\n===== DNN 결과 =====")
print(f"총 파라미터 수: {model_dnn.count_params():,}")
print(f"실제 학습 Epoch: {actual_epochs_dnn}")
print(f"복원 모델 Train Accuracy: {train_acc_dnn:.4f}")
print(f"복원 모델 Validation Accuracy: {val_acc_dnn:.4f}")
print(f"복원 모델 Train Loss: {train_loss_dnn:.4f}")
print(f"복원 모델 Validation Loss: {val_loss_dnn:.4f}")
print(f"학습 중 최고 Validation Accuracy: {best_val_acc_dnn:.4f} (Epoch {best_val_acc_epoch_dnn})")
print(f"학습 중 최저 Validation Loss: {best_val_loss_dnn:.4f} (Epoch {best_val_loss_epoch_dnn})")
print(f"학습 시간: {elapsed_dnn:.1f}초")

Epoch 1/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 404ms/step - accuracy: 0.3411 - loss: 1.6682 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 410ms/step - accuracy: 0.3344 - loss: 1.1535 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 402ms/step - accuracy: 0.3333 - loss: 1.0987 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 24s 423ms/step - accuracy: 0.3217 - loss: 1.0987 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 25s 440ms/step - accuracy: 0.3333 - loss: 1.0987 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 6/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 31s 551ms/step - accuracy: 0.3333 - loss: 1.0987 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 7/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 408ms/step - accuracy: 0.3333 - loss: 1.0987 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 8/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 23s 399ms/step - accuracy: 0.3167 - loss: 1.0987 - val_accu

## 비교 실험: DNN vs CNN 결과 분석

- 모델 구조: Rescaling → Flatten → Dense(32) → Dropout(0.5) → Dense(3)
- 총 파라미터: 1,572,995개
- 실제 학습 Epoch: 8
- 복원 모델 train_accuracy: 0.3333
- 복원 모델 val_accuracy: 0.3333
- 복원 모델 val_loss: 1.0986
- 학습 중 최고 val_accuracy: 0.3333 (Epoch 1)
- 학습 중 최저 val_loss: 1.0986 (Epoch 1)

- **CNN(실험 10)과 비교**:
  - 파라미터: DNN 1,572,995개 / CNN 15,491개 (DNN이 약 101.5배 많음)
  - val_accuracy: DNN 0.3333 / CNN 0.8644 (CNN이 53.11%p 높음)
  - val_loss: DNN 1.0986 / CNN 0.3261
  - DNN은 Epoch 1 이후 val_loss가 개선되지 않아 EarlyStopping으로 Epoch 8에서 학습이 종료됨

- **분석**:
  - DNN은 128×128×3 이미지를 Flatten하여 49,152개의 값으로 변환한 뒤 Dense층에 연결하면서 CNN보다 훨씬 많은 파라미터를 사용함
  - 이번 DNN 실행에서는 Train/Validation Accuracy가 모두 약 33.33%에 머물렀고, Validation Loss도 약 1.0986에서 개선되지 않아 세 클래스를 구분하는 유의미한 학습으로 이어지지 않음
  - 반면 CNN은 Conv2D를 통해 이미지의 공간적 관계를 유지하면서 특징을 추출하고, 훨씬 적은 파라미터로 더 높은 Validation 성능을 보임

- **결론**: 동일한 재분배 데이터를 사용한 이번 비교에서 Flatten + Dense 기반 DNN은 CNN보다 약 101.5배 많은 파라미터를 사용했지만 유의미한 분류 성능으로 이어지지 않았음. 반면 CNN은 15,491개의 파라미터로 0.8644의 Validation Accuracy를 보였으며, 이번 작물 이미지 분류 실험에서는 Convolution을 이용해 공간적 특징을 추출하는 CNN 구조가 더 효과적인 결과를 보였음.

# ===================================
# 4부: Keras vs PyTorch 비교
# ===================================

지금까지 Keras로 구현하고 검증한 최종 CNN 모델 구조(실험10, 파라미터 15,491개)가
다른 프레임워크에서도 동일하게 구현 가능한지, 그리고 코드 스타일이 어떻게
다른지 비교하기 위해, 동일한 재분배 데이터를 PyTorch로 불러와
동일한 구조의 모델을 학습시켜 비교함.

In [ ]:
import torch
print(torch.__version__)

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()   # 0~255 -> 0~1 자동 변환
])

train_dataset_pt = datasets.ImageFolder(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\train",
    transform=transform
)
val_dataset_pt = datasets.ImageFolder(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\val",
    transform=transform
)

train_loader_pt = DataLoader(train_dataset_pt, batch_size=32, shuffle=True)
val_loader_pt = DataLoader(val_dataset_pt, batch_size=32, shuffle=False)

print("클래스 목록:", train_dataset_pt.classes)

In [ ]:
class CropClassifierPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.gap = nn.AdaptiveAvgPool2d(1)   # Keras의 GlobalAveragePooling2D와 동일
        self.fc1 = nn.Linear(32, 32)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, 3)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.gap(x)
        x = x.view(x.size(0), -1)   # Keras의 Flatten과 유사한 역할 (GAP 결과를 1차원으로)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)   # softmax는 loss 함수 안에 포함되어 여기선 생략

model_pt = CropClassifierPT()
print(model_pt)

total_params = sum(p.numel() for p in model_pt.parameters())
print(f"총 파라미터: {total_params:,}개")

In [ ]:
# =========================================================
# PyTorch 학습 함수
# =========================================================
def train_pytorch(model, train_loader, val_loader, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Keras의 sparse_categorical_crossentropy에 대응
    criterion = nn.CrossEntropyLoss()

    # Keras와 동일하게 Adam 사용
    optimizer = torch.optim.Adam(model.parameters())

    history = {
        "accuracy": [],
        "val_accuracy": [],
        "loss": [],
        "val_loss": []
    }

    for epoch in range(epochs):

        # Train
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # 이전 Gradient 초기화
            optimizer.zero_grad()

            # Forward
            outputs = model(images)

            # Loss 계산
            loss = criterion(outputs, labels)

            # Backpropagation
            loss.backward()

            # Weight Update
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss = train_loss / train_total
        train_accuracy = train_correct / train_total

        # Validation
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / val_total
        val_accuracy = val_correct / val_total

        # 결과 저장
        history["accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"loss: {train_loss:.4f} - "
            f"accuracy: {train_accuracy:.4f} - "
            f"val_loss: {val_loss:.4f} - "
            f"val_accuracy: {val_accuracy:.4f}"
        )

    return history

In [ ]:
# DataLoader를 num_workers=0으로 되돌리기 (안전하게)
train_loader_pt = DataLoader(train_dataset_pt, batch_size=32, shuffle=True, num_workers=0)
val_loader_pt = DataLoader(val_dataset_pt, batch_size=32, shuffle=False, num_workers=0)

# 모델도 새로 초기화 (혹시 이전 중단으로 이상한 상태가 남았을 수 있으니 깨끗하게)
model_pt = CropClassifierPT()

result_pt = train_pytorch(model_pt, train_loader_pt, val_loader_pt, epochs=3)

## 비교 실험: Keras vs PyTorch

- Keras 실험 10에서 사용한 CNN 구조(Conv 16-32-32, GAP, Dense32, Dropout0.5)를 PyTorch로 다시 구현함
- PyTorch 모델의 총 파라미터 수는 15,491개로 Keras 모델과 동일함을 확인
- PyTorch에서는 마지막 출력층에 Softmax를 별도로 적용하지 않고 `CrossEntropyLoss`를 사용함
- 3 epoch 학습 결과:
  - train_accuracy: 0.3811 → 0.5711
  - val_accuracy: 0.5600 → 0.6178
  - train_loss: 1.0845 → 0.8905
  - val_loss: 0.9743 → 0.8108
- **코드 구조 차이**:
  - Keras에서는 `compile()`과 `fit()`을 통해 학습 과정을 높은 수준에서 처리함
  - PyTorch에서는 `optimizer.zero_grad()` → Forward → Loss 계산 → `loss.backward()` → `optimizer.step()`의 학습 과정을 직접 작성함
  - Validation에서도 `model.eval()`과 `torch.no_grad()`를 직접 설정함
- **결론**: Keras에서 사용한 CNN 구조를 PyTorch에서도 동일한 파라미터 수의 모델로 구현할 수 있음을 확인함. 또한 직접 학습 루프를 작성하면서 순전파, Loss 계산, 역전파, 가중치 업데이트가 어떤 순서로 이루어지는지 코드 수준에서 확인함. 이번 비교의 목적은 두 프레임워크의 성능 우위를 판단하는 것이 아니라, 동일한 CNN 구조를 서로 다른 방식으로 구현하고 학습 과정의 차이를 이해하는 데 있음.

In [ ]:
print(train_ds_v2)

# ===================================
# 5부: 최종 모델 저장
# ===================================

지금까지의 실험 결과, 재분배 데이터에서 학습한 모델(실험10,
파라미터 15,491개, val_accuracy 91.78%)을 최종 모델로 확정하고
파일로 저장함.

In [ ]:
import os

model_dir = r"C:\Users\AISW_203_103\Desktop\crop-classifier\model"
os.makedirs(model_dir, exist_ok=True)

model_v2_epoch50.save(os.path.join(model_dir, "crop_classifier_final.keras"))
print("모델 저장 완료:", os.path.join(model_dir, "crop_classifier_final.keras"))

# ===================================
# 6부: 예측 테스트
# ===================================

저장된 최종 모델을 불러와, 실제 이미지에 대한 예측이 정상적으로
수행되는지 확인함. 모델이 예측한 클래스와 확신도(confidence)를
이미지와 함께 시각화하여, 학습된 모델의 실제 동작을 검증함.

In [ ]:
from tensorflow import keras

loaded_model = keras.models.load_model(r"C:\Users\AISW_203_103\Desktop\crop-classifier\model\crop_classifier_final.keras")
loaded_model.summary()

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

class_names = ['cabbage', 'cucumber', 'pepper']
class_names_kor = {'cabbage': '배추', 'cucumber': '오이', 'pepper': '고추'}

def predict_image(image_path, model):
    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(prediction)
    predicted_class = class_names_kor[class_names[predicted_idx]]
    confidence = np.max(prediction) * 100

    plt.imshow(img)
    plt.title(f"예측: {predicted_class} ({confidence:.1f}%)")
    plt.axis('off')
    plt.show()

    return predicted_class, confidence

In [ ]:
import os

test_folder = r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\val\pepper"
sample_file = os.listdir(test_folder)[0]
predict_image(os.path.join(test_folder, sample_file), loaded_model)

In [ ]:
# 배추, 오이도 테스트
test_folder_cabbage = r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\val\cabbage"
sample_file = os.listdir(test_folder_cabbage)[0]
predict_image(os.path.join(test_folder_cabbage, sample_file), loaded_model)

In [ ]:
test_folder_cucumber = r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\val\cucumber"
sample_file = os.listdir(test_folder_cucumber)[0]
predict_image(os.path.join(test_folder_cucumber, sample_file), loaded_model)

In [ ]:
sample_files = os.listdir(test_folder_cabbage)[21:24]
for f in sample_files:
    predict_image(os.path.join(test_folder_cabbage, f), loaded_model)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_true = []
y_pred = []

for images, labels in val_ds_v2:
    predictions = model_v2_epoch50.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)
class_names_list = ['cabbage', 'cucumber', 'pepper']

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names_list, yticklabels=class_names_list, cmap='Blues')
plt.xlabel('예측')
plt.ylabel('실제')
plt.title('Confusion Matrix (최종 확정 모델)')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names_list))

## Confusion Matrix 종합 해석 (최종 모델 기준)

- 전체 정확도(Accuracy): 85.56%

- 클래스별 Recall:
  cabbage 0.92, cucumber 0.84, pepper 0.81로,  cabbage의 Recall이 가장 높았고 pepper가 가장 낮게 나타남
- 가장 빈번한 오분류:  pepper → cucumber 28건,  cucumber → pepper 18건으로,  cucumber와 pepper 사이에서 양방향 혼동이 나타남
- cucumber의 Precision은 0.76으로 세 클래스 중 가장 낮음.  실제 cabbage 11건과 pepper 28건이 cucumber로 잘못 예측되어,
  다른 클래스를 cucumber로 예측하는 경우가 상대적으로 많이 나타남
- cabbage는 Precision 0.95, Recall 0.92, F1-score 0.94로  세 클래스 중 가장 높은 분류 성능을 보임.
  실제 cabbage 150장 중 138장을 정확하게 분류했으며,  11장은 cucumber, 1장은 pepper로 오분류됨
- pepper는 실제 150장 중 121장을 정확하게 분류했으며,  28장이 cucumber로 오분류되어  가장 큰 단일 오분류 패턴이 pepper → cucumber에서 나타남

In [ ]:
# val_ds_v2가 실제로 이미지를 어떻게 처리하는지 직접 확인
for images, labels in val_ds_v2.take(1):
    print("이미지 값 범위:", images.numpy().min(), "~", images.numpy().max())
    print("이미지 shape:", images.shape)
    print("이미지 dtype:", images.dtype)
    break

In [ ]:
# 추론 확인용 배추 이미지 경로
cabbage_folder = os.path.join(processed_path, "val", "cabbage")

target_files = sorted([
    f for f in os.listdir(cabbage_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print("cabbage folder:", cabbage_folder)
print("이미지 수:", len(target_files))
print("확인할 파일:", target_files[:4])

In [ ]:
from tensorflow import keras
import numpy as np

img_path = os.path.join(cabbage_folder, target_files[0])  # 아까 그 파일

# 방법 1: PIL 방식 (predict_image 함수가 쓰는 것)
img_pil = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
array_pil = np.array(img_pil)

# 방법 2: Keras/TensorFlow 방식 (val_ds_v2가 쓰는 것)
img_tf = keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
array_tf = keras.utils.img_to_array(img_tf)

print("PIL 방식 픽셀 일부:", array_pil[0,0], array_pil[64,64])
print("TF 방식 픽셀 일부:", array_tf[0,0], array_tf[64,64])
print("두 배열이 완전히 같은가?", np.array_equal(array_pil, array_tf))

In [ ]:
from tensorflow import keras

def predict_image(image_path, model):
    # Keras의 공식 로딩 방식으로 통일 (val_ds_v2와 동일한 방식)
    img = keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(prediction)
    predicted_class = class_names_kor[class_names[predicted_idx]]
    confidence = np.max(prediction) * 100

    plt.imshow(img)
    plt.title(f"예측: {predicted_class} ({confidence:.1f}%)")
    plt.axis('off')
    plt.show()

    return predicted_class, confidence

In [ ]:
predict_image(os.path.join(cabbage_folder, target_files[0]), loaded_model)
predict_image(os.path.join(cabbage_folder, target_files[1]), loaded_model)
predict_image(os.path.join(cabbage_folder, target_files[2]), loaded_model)
predict_image(os.path.join(cabbage_folder, target_files[3]), loaded_model)

## 예측 파이프라인 검증 중 발견한 전처리 불일치 문제

초기 예측 테스트에서 배추 이미지가 반복적으로 오이로 오분류되는 현상을 확인함.  
Confusion Matrix에서는 cabbage recall이 0.92로 나타났기 때문에, 모델의 전체적인 배추 분류 성능과 별개로 예측 함수의 전처리 과정을 점검함.

- **문제 확인**: `predict_image()`에서는 PIL의 `Image.resize()`를 사용했지만, 학습/평가에서는 Keras의 이미지 로딩 방식을 사용하고 있었음. 동일 이미지를 두 방식으로 처리했을 때 픽셀 값에 차이가 나타남을 확인함 (예: [42, 72, 14] vs [79, 105, 34])
- **수정**: 예측 함수의 이미지 로딩 및 리사이즈 방식을 `keras.utils.load_img()`를 사용하는 방식으로 변경
- **검증 결과**: 기존에 오분류되었던 배추 이미지 4장을 수정된 예측 함수로 다시 테스트한 결과 모두 배추로 예측됨 (확신도 83~99%)
- **결론**: 학습·평가와 실제 예측 과정에서 서로 다른 이미지 전처리 방식이 사용되고 있음을 발견하고 이를 통일함. 수정 후 기존 오분류 이미지의 예측 결과가 변경되는 것을 확인하면서, 실제 추론 단계에서도 학습·평가와 일관된 전처리 파이프라인을 사용하는 것이 중요함을 확인함.

==================================================================================================

In [ ]:
import os

for root, dirs, files in os.walk(r"C:\Users\AISW_203_103"):
    if "crop_classifier_final_600.keras" in files:
        print(os.path.join(root, "crop_classifier_final_600.keras"))

In [ ]:
from tensorflow import keras

# 1. 저장된 최종 모델 불러오기
model = keras.models.load_model(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\model\crop_classifier_final_600.keras"
)

print("모델 불러오기 완료")

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

val_ds_v2 = keras.utils.image_dataset_from_directory(
    r"C:\Users\AISW_203_103\Desktop\crop-classifier\data\processed_v2\val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds_v2, verbose=1)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_true = []
y_pred = []

for images, labels in val_ds_v2:
    # 저장된 최종 모델로 예측
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)
class_names_list = ['cabbage', 'cucumber', 'pepper']

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=class_names_list,
    yticklabels=class_names_list,
    cmap='Blues'
)

plt.xlabel('예측')
plt.ylabel('실제')
plt.title('Confusion Matrix (최종 확정 모델)')
plt.show()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names_list
    )
)